# **EDA Notebook**



---
## 0. Setup Environment

In [ ]:
# Do not modify this code
%pip install uts_36106_core==0.0.3

from uts_36106_core.folders import *
from uts_36106_core.ipyrenders import *
import os

at = AtFolder(
    course_code=36106,
    assignment="AT3",
)
at.mount_gdrive()
at.create_folders()
if at.is_colab:
  os.chdir(f'{at.folder_path}')

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
group_name = "Group 52"
student_name = "Andres Ethorimn Lobarb"
student_id = "26029165"

In [ ]:
# Do not modify this code
print_tile(size="h1", key='group_name', value=group_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# Do not modify this code
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

In [ ]:
# No additional packages: pandas, numpy, scipy and altair are available on Colab.

### 0.b Import Packages

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
from scipy.stats import norm, spearmanr
from IPython.display import display

In [ ]:
# Utils

def column_profile(df):
    """One row per column: type, missing count and share, distinct values."""
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_share": df.isna().mean().round(4),
        "distinct": df.nunique(),
    })


def monthly_mean_change(balances, account_col, date_col, balance_col):
    """Mean month-on-month balance change across accounts, recomputed from balances."""
    ordered = balances.sort_values([account_col, date_col])
    change = ordered.groupby(account_col)[balance_col].diff()
    frame = ordered.assign(change=change).dropna(subset=["change"])
    frame["month"] = pd.to_datetime(frame[date_col]).dt.to_period("M")
    return frame, frame.groupby("month")["change"].mean().rename("mean_change").reset_index()


def lagged_indicators(economics, date_col, columns, lag_months):
    """Indicators keyed by the month they are used in: the row dated `lag_months` earlier."""
    out = economics[[date_col] + columns].copy()
    out["month"] = pd.to_datetime(out[date_col]).dt.to_period("M") + lag_months
    return out.drop(columns=date_col)


def spearman_summary(x, y, label):
    """Spearman correlation with a Fisher-z 95% interval and a size label."""
    mask = x.notna() & y.notna()
    rho, p_value = spearmanr(x[mask], y[mask])
    n = int(mask.sum())
    half_width = 1.96 / np.sqrt(n - 3)
    low, high = np.tanh(np.arctanh(rho) - half_width), np.tanh(np.arctanh(rho) + half_width)
    size = abs(rho)
    magnitude = "negligible" if size < 0.1 else "small" if size < 0.3 else "medium" if size < 0.5 else "large"
    return pd.Series({"feature": label, "n": n, "rho": round(rho, 4), "ci_low": round(low, 4),
                      "ci_high": round(high, 4), "p_value": p_value, "magnitude": magnitude})


def holm_adjust(p_values):
    """Holm step-down adjustment of a list of p-values (family-wise error control)."""
    order = np.argsort(p_values)
    m = len(p_values)
    adjusted = np.empty(m)
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * p_values[index])
        adjusted[index] = min(running, 1.0)
    return adjusted


def indicator_scatter(table, indicator, x_title):
    """Monthly mean balance change against a lagged indicator, one point per month."""
    data = table[["month_label", indicator, "mean_change"]]  # Period values are not JSON-serialisable
    return alt.Chart(data).mark_circle(size=60, color="#2a78d6").encode(
        x=alt.X(f"{indicator}:Q", title=x_title, scale=alt.Scale(zero=False)),
        y=alt.Y("mean_change:Q", title="Mean monthly balance change across accounts ($)", scale=alt.Scale(zero=False)),
        tooltip=["month_label", indicator, alt.Tooltip("mean_change:Q", format=".1f")],
    ).properties(width=420, height=260)


def qq_frame(values):
    """Sample quantiles against normal quantiles (all points up to 500 values, else 199 levels).

    Robust standardisation (median, IQR / 1.349) keeps a few extreme values from flattening the plot;
    for normal data it equals the usual standardisation."""
    values = pd.Series(values).dropna()
    n = len(values)
    levels = (np.arange(1, n + 1) - 0.5) / n if n <= 500 else np.linspace(0.005, 0.995, 199)
    scale = (values.quantile(0.75) - values.quantile(0.25)) / 1.349
    scale = scale if scale > 0 else values.std()  # IQR is 0 when most values are equal
    z = (values - values.median()) / scale
    return pd.DataFrame({"theoretical": norm.ppf(levels), "sample": np.quantile(z, levels)})


def distribution_charts(values, label, n_bins, tail):
    """Histogram (the `tail` share at each end not drawn) beside a normal QQ plot, with shape statistics."""
    values = pd.Series(values).dropna()
    low, high = values.quantile(tail), values.quantile(1 - tail)
    shown = values[(values >= low) & (values <= high)]
    counts, edges = np.histogram(shown, bins=n_bins)
    hist = alt.Chart(pd.DataFrame({"start": edges[:-1], "end": edges[1:], "count": counts})).mark_bar(
        color="#2a78d6").encode(x=alt.X("start:Q", bin="binned", title=label), x2="end:Q",
                                y=alt.Y("count:Q", title="Count")).properties(width=300, height=220)
    diagonal = pd.DataFrame({"theoretical": [-3, 3], "sample": [-3, 3]})
    qq = (alt.Chart(qq_frame(values)).mark_point(color="#2a78d6", size=14).encode(
            x=alt.X("theoretical:Q", title="Normal quantile"), y=alt.Y("sample:Q", title="Robust-standardised sample quantile"))
          + alt.Chart(diagonal).mark_line(color="#898781", strokeDash=[4, 4]).encode(x="theoretical:Q", y="sample:Q")
          ).properties(width=300, height=220)
    print(f"{label}: n = {len(values)}, skewness = {values.skew():.2f}, excess kurtosis = {values.kurt():.2f}, "
          f"histogram omits {len(values) - len(shown)} values outside the {tail:.1%} to {1 - tail:.1%} quantiles")
    return alt.hconcat(hist, qq)

In [ ]:
# Constants
# Each economic row is dated the first day of its month. A figure for a month is published
# after that month, so the row used for a month is the previous month's (one-month lag).
LAG_MONTHS = 1
INDICATORS = ["inflation_rate", "interest_rate", "unemployment_rate", "consumer_sentiment",
              "loan_stress_index", "spending_factor", "housing_price_index", "equity_market_index"]

---
## B. Data Understanding

**Approach.** The indicator table first (B.1), then three indicators chosen in advance for their meaning for deposits (B.2 to B.4), then a corrected sweep of all indicators. Findings come only from the outputs above them; how the economy enters the forecast is decided after all three EDA notebooks.

In [ ]:
# Do not modify this code
try:
  df = pd.read_csv(at.folder_path / "economics.csv")
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [ ]:
df["economic_date"] = pd.to_datetime(df["economic_date"])
print("Rows, columns:", df.shape)
print("From", df["economic_date"].min().date(), "to", df["economic_date"].max().date(),
      "| all dated the first of a month:", bool((df["economic_date"].dt.day == 1).all()))
print("Months missing in the range:",
      len(pd.date_range(df["economic_date"].min(), df["economic_date"].max(), freq="MS").difference(df["economic_date"])))
display(column_profile(df))
display(df.describe().T.round(3))
print("economic_cycle:", df["economic_cycle"].value_counts().to_dict())
print("recession_flag:", df["recession_flag"].value_counts().to_dict())
cycle_runs = (df["economic_cycle"] != df["economic_cycle"].shift()).cumsum()
print("Consecutive runs of the same cycle phase (months):",
      df.groupby(cycle_runs)["economic_cycle"].agg(["first", "size"]).values.tolist()[:12])

In [ ]:
standardised = df[["economic_date"] + INDICATORS].copy()
standardised[INDICATORS] = (standardised[INDICATORS] - standardised[INDICATORS].mean()) / standardised[INDICATORS].std()
long = standardised.melt("economic_date", var_name="indicator", value_name="z")
long["indicator"] = long["indicator"].str.replace("_", " ").str.capitalize()  # readable legend labels
alt.Chart(long).mark_line().encode(
    x=alt.X("economic_date:T", title="Month"),
    y=alt.Y("z:Q", title="Indicator in standard deviations from its mean"),
    color=alt.Color("indicator:N", title="Indicator"),
    tooltip=["economic_date:T", "indicator", alt.Tooltip("z:Q", format=".2f")],
).properties(width=620, height=260)

In [ ]:
dataset_insights = """
72 complete monthly rows, 2020-01 to 2025-12. Interest rate 2.36% to 3.44%, unemployment 4.59% to
5.42%, consumer sentiment 69.1 to 99.0. Sentiment and <spending_factor> drop in a step from
January 2023 to July 2024 (chart).

<economic_cycle> is four fixed 18-month blocks and <recession_flag> is always 0: neither carries
information.

Each month's balance change is compared with the previous month's figures (publication lag):
47 months, so intervals are wide.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `interest_rate`

In [ ]:
# Complementary load: how balances moved, month by month, across all accounts
balances = pd.read_csv(at.folder_path / "account_balance_history.csv",
                       usecols=["account_id", "balance_date", "balance"])
account_months, monthly = monthly_mean_change(balances, "account_id", "balance_date", "balance")
table = monthly.merge(lagged_indicators(df, "economic_date", INDICATORS, LAG_MONTHS), on="month", how="inner")
table["month_label"] = table["month"].astype(str)
print("Months with a balance change and a lagged indicator:", len(table),
      "| from", table["month_label"].iloc[0], "to", table["month_label"].iloc[-1])

distribution_charts(table["interest_rate"], "Interest rate, previous month (%)", 12, 0)

In [ ]:
display(table["mean_change"].describe().round(1).to_frame("mean monthly change across accounts ($)"))

interest_test = spearman_summary(table["interest_rate"], table["mean_change"], "interest_rate")
display(interest_test.to_frame("Spearman, previous month's interest rate vs mean balance change"))

In [ ]:
indicator_scatter(table, "interest_rate", "Interest rate, previous month (%)")

In [ ]:
feature_1_insights = """
Target: mean monthly balance change across accounts, $181 to $293 over 47 months (mean $233).
Interest rate is mildly right-skewed (plots above).

Spearman rho = 0.33 (95% interval 0.05 to 0.57): medium positive association, with a wide scatter
(chart).
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `unemployment_rate`

In [ ]:
distribution_charts(table["unemployment_rate"], "Unemployment rate, previous month (%)", 12, 0)

In [ ]:
unemployment_test = spearman_summary(table["unemployment_rate"], table["mean_change"], "unemployment_rate")
display(unemployment_test.to_frame("Spearman, previous month's unemployment rate vs mean balance change"))

In [ ]:
indicator_scatter(table, "unemployment_rate", "Unemployment rate, previous month (%)")

In [ ]:
feature_2_insights = """
Unemployment is left-skewed (plots above).

Spearman rho = -0.36 (95% interval -0.59 to -0.08): medium negative association; months below 4.8%
unemployment all show mean changes of about $238 or more (chart).
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `consumer_sentiment`

In [ ]:
distribution_charts(table["consumer_sentiment"], "Consumer sentiment, previous month (index)", 12, 0)

In [ ]:
sentiment_test = spearman_summary(table["consumer_sentiment"], table["mean_change"], "consumer_sentiment")
display(sentiment_test.to_frame("Spearman, previous month's consumer sentiment vs mean balance change"))

In [ ]:
indicator_scatter(table, "consumer_sentiment", "Consumer sentiment, previous month (index)")

In [ ]:
feature_n_insights = """
Sentiment falls into two groups of months (about 69 to 73 and 90 to 94; plots above).

Spearman rho = 0.33 (95% interval 0.05 to 0.56): medium positive association, seen as two clusters
(chart).

The three pre-chosen associations hold after Holm correction (p 0.037 to 0.045); in the sweep of
all eight indicators none does (smallest p 0.098). The month explains a 0.000035 share of
account-level change variance: the economy relates to the bank-wide average, almost not to one
account's balance.
"""

In [ ]:
# Do not modify this code
print_tile(size="h3", key='feature_n_insights', value=feature_n_insights)

In [ ]:
# Holm correction within the three indicators chosen in advance (B.2 to B.4)
chosen = pd.DataFrame([interest_test, unemployment_test, sentiment_test]).set_index("feature")
chosen["p_holm"] = holm_adjust(chosen["p_value"].to_numpy())
display(chosen)

# Exploratory sweep across every indicator, corrected for all eight tests (Holm), and how much
# account-level change the month can explain at most: the share of the variance of account-month
# changes that lies between months.
sweep = pd.DataFrame([spearman_summary(table[c], table["mean_change"], c) for c in INDICATORS]).set_index("feature")
sweep["p_holm"] = holm_adjust(sweep["p_value"].to_numpy())
display(sweep)

grand_mean = account_months["change"].mean()
month_means = account_months.groupby("month")["change"].transform("mean")
between = ((month_means - grand_mean) ** 2).sum()
total = ((account_months["change"] - grand_mean) ** 2).sum()
print("Account-level monthly change, standard deviation ($):", round(account_months["change"].std(), 1))
print("Share of account-month change variance between months (eta squared):", round(between / total, 6))

## C. Ethical Consideration of Features


In [ ]:
# No computation: the economic indicators carry no customer attribute; see the text below.

In [ ]:
ethical_considerations = """
Indicators take the same value for every account, so they cannot treat customers differently by
themselves. Downturns hit groups unequally, so errors are audited by group; a macro figure never
explains or triggers action on one customer's situation.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='ethical_considerations', value=ethical_considerations)